In [1]:
from kdrag.all import *

BV16 = smt.BitVecSort(16)
# Would kd.reflect.struct be nice here?
State = kd.Struct("NandState",
                  ("pc", BV16),
                  ("rom", smt.ArraySort(BV16, BV16)), # Or in decoded form?
                  ("ram", smt.ArraySort(BV16, BV16)),
                  ("M", BV16),
                  ("A", BV16)
)

I'd also want a python model of the simplest riscv cpu.
Then verify _that_ against riscv formal via hwbmc or yosys

I could compile something to x86 and use existing assembly verifier? But there kind of aren't any



model_check
bisim


We are kind of spanning both a time and space refinement of single instruction cycle and 


In [ ]:
from kdrag.all import *
from kdrag.contrib.nand2tetris import *

def model_check(prog : list[int], insns=10, init=None):
    if init is None:
        init = init_state(prog)
    states = [(insns, [], init)]
    while states:
        insns, s = states.pop()
        s1 = step(s)
        for 


In [2]:
@kd.reflect.reflect
def nand2step(st : State) -> State:
    # a single step of the nand2tetris cpu
    insn = st.rom[st.pc]
    D = st.ram[st.A]
    if smt.Extract(0,0,insn) == 1:
        # A-instruction
        return st._replace(pc=st.pc + 1, A=smt.ZeroExt(1,smt.Extract(15,1,insn)))
    else:
        dst = smt.Extract(5,3,insn)
        comp = smt.Extract(12,6,insn)
        jmp = smt.Extract(15,13,insn)
        # compute the ALU output
        if comp == 0b101010:
            out = 0
        elif comp == 0b111111:
            out = 1
        elif comp == 0b111010:
            out = -1
        elif comp == 0b001100:
            out = D
        elif comp == 0b110000:
            out = st.A
        elif comp == 0b001101:
            out = ~D
        elif comp == 0b110001:
            out = ~st.A
        elif comp == 0b001111:
            out = -D
        elif comp == 0b110011:
            out = -st.A
        elif comp == 0b011111:
            out = D + 1
        elif comp == 0b110111:
            out = st.A + 1
        elif comp == 0b001110:
            out = D - 1
        elif comp == 0b110010:
            out = st.A - 1
        elif comp == 0b000010:
            out = D + st.A
        elif comp == 0b010011:
            out = D - st.A
        elif comp == 0b000111:
            out = st.A - D
        elif comp == 0b000000:
            out = D & st.A
        elif comp == 0b010101:
            out = D | st.A
        #else:
        #    # Hmm. Maybe I should support raise in statically impossible cases.
        #    # and asserts as tagged properties?
        #    raise ValueError("Invalid comp field")
        # hmm this is a merge point.
        # I could lift this as a function

        # write the output to the destinations
        new_st = st._replace(pc=st.pc + 1)
        if dst[0] == 1:
            new_st = new_st._replace(M=out)
        if dst[1] == 1:
            new_st = new_st._replace(D=out)
        if dst[2] == 1:
            new_st = new_st._replace(A=out)
        # handle the jump
        if jmp == 0b000:
            return new_st
        elif jmp == 0b001 and out > 0:
            return new_st._replace(pc=st.A)
        elif jmp == 0b010 and out == 0:
            return new_st._replace(pc=st.A)
        elif jmp == 0b011 and out >= 0:
            return new_st._replace(pc=st.A)
        elif jmp == 0b100 and out < 0:
            return new_st._replace(pc=st.A)
        elif jmp == 0b101 and out != 0:
            return new_st._replace(pc=st.A)
        elif jmp == 0b110 and out <= 0:
            return new_st._replace(pc=st.A)
        elif jmp == 0b111:
            return new_st._replace(pc=st.A)
        else:
            return new_st

AssertionError: ('If statement condition is not exhaustive: ', 'if comp == 21:\n    out = D | st.A', {'st': NandState(0,
          Store(K(BitVec(16), 3520), 0, 64),
          K(BitVec(16), 32823),
          0,
          0), 'nand2step': nand2step, 'insn': 64, 'D': 32823, 'dst': 0, 'comp': 1, 'jmp': 0}, [st = NandState(0,
                Store(K(BitVec(16), 3520), 0, 64),
                K(BitVec(16), 32823),
                0,
                0),
 Ext = [else -> 8]])

In [5]:
x = smt.Const("x", BV16)
smt.Extract(0,0,x).size()

1

icestudio
https://github.com/Obijuan/nand2tetris-icestudio

My verilog

Yeah, the instruction adt just is boiler plate.


Could have 
kd.precond
kd.postcond

Just use eval for expressions? I'm gaining if-then-else, and, or, but so what
Homoiconicity with normal python code I am not so sure is exactly the goal.
Use a generator effects loop?
Monadic code. Pulse clone. A heap as state.
A list is reference to a list of references
subtypes

for loops would enable bounded model checking, which would be neat.

else:
    raise Error() 
is a nice looking pattern. So maybe the thing should be in a result monad

Locals
Return
Error

do unchain. How do they desugar early return. It _is_ pretty nice.
Maybe I _shouldn't turn `int` into IntSort. Hmm.

T = Generic
Expr[S] = NewType



# Latte 2026

https://github.com/ethanuppal/marlin rust verilator setup?

hardware languages
https://spade-lang.org/
https://veryl-lang.org/

https://dynamatic.epfl.ch/ hls

# amaranth


https://github.com/MikePopoloski/slang pyslang. It has a evaluator?


In [1]:
import pyslang
session = pyslang.ScriptSession()
session.eval("logic bit_arr [16] = '{0:1, 1:1, 2:1, default:0};")
result = session.eval("bit_arr.sum with ( int'(item) );")
print(result)

3
